# CLaRa — Apple Native Evaluation (Kaggle T4)

> **Paper:** *CLaRa: Bridging Retrieval and Generation with Continuous Latent Reasoning*  
> He et al., Apple / University of Edinburgh — February 2026  
> **Checkpoint:** `tokiggle/clara-7b-e2e-4q` (Kaggle dataset)

---

## Overview

This notebook evaluates the Apple CLaRa-7B-E2E pretrained checkpoint using **Apple's own `modeling_clara.py`** via `trust_remote_code=True`, with 4-bit NF4 quantization to fit on T4 GPUs.

### Why Apple Native instead of the repo's reimplementation?

The repo's `models/clara_model.py` had 5 critical architectural mismatches:

| Bug | Impact |
|-----|--------|
| Wrong compression (embedding concat vs memory token injection) | Memory tokens contain zero document info |
| Wrong adapter names (`compressor/query/generator` vs `encoder/query_reasoner/decoder`) | Incorrect weight loading |
| Wrong LoRA targets (attention-only vs `all-linear`) | ~50% of trained weights dropped |
| Wrong prompt format (no system prompt / memory token placeholders) | Model never saw this format |
| Missing special token embeddings from `decoder_first_last_layers.pth` | Random embeddings for MEM tokens |

Using Apple's original code bypasses all of these → **correct evaluation results**.

### Notebook Structure

```
Section 0 — Environment Setup (clone repo, install deps)
Section 1 — Quick Test (5 SQuAD samples → verify pipeline)
Section 2 — Manual Inference (hand-crafted Q&A examples)
Section 3 — Full Evaluation (500 samples × TriviaQA + SQuAD)
Section 4 — Results Summary
```

---
## Section 0 — Environment Setup

**Run once.** Clone the repo and install dependencies, then **restart the kernel** before running any further cells.

In [1]:
! ls ../../root/.cache

matplotlib  node-gyp


In [2]:
import shutil
import os

# Kill the hidden HF code cache
cache_path = "/root/.cache/huggingface/modules/transformers_modules"
if os.path.exists(cache_path):
    print("Nuking stale code cache...")
    shutil.rmtree(cache_path)
    print("Cache destroyed. Next load will use fresh files.")
else:
    print("Cache was already empty.")

Cache was already empty.


In [3]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-A  │  Clone repository & install dependencies
# Run ONCE. After this cell completes → Session > Restart & Run All (skip 0-A).
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess, sys

REPO_URL  = "https://github.com/Duy-Tuyen/introml-clara-implementation.git"
REPO_NAME = "introml-clara-implementation"
REPO_ROOT = f"/kaggle/working/{REPO_NAME}"

print("[1/3] Cloning repository...")
subprocess.run(["rm", "-rf", REPO_ROOT], check=True)
subprocess.run(["git", "clone", "-b", "feature-tuyen2", REPO_URL, REPO_ROOT], check=True)

print("[2/3] Installing dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     f"{REPO_ROOT}/requirements.txt", "-q"],
    check=True,
)

print("[3/3] Running setup script...")
subprocess.run([sys.executable, f"{REPO_ROOT}/setup_env.py"], check=True)

print("\n Setup complete. Please RESTART the kernel before continuing.")

[1/3] Cloning repository...


Cloning into '/kaggle/working/introml-clara-implementation'...


[2/3] Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.6/251.6 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.8/119.8 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.9/167.9 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.4/324.4 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.3.1 which is incompatible.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.3.1 which is incompatible.


[3/3] Running setup script...
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 18.6 MB/s eta 0:00:00

Patched bitsandbytes CUDA 12.8 → libbitsandbytes_cuda124_nocublaslt.so
Torch: 2.3.1+cu121 | CUDA: 12.1
GPU: Tesla T4
Môi trường Kaggle đã sẵn sàng. Hãy RESTART KERNEL rồi chạy tiếp.

 Setup complete. Please RESTART the kernel before continuing.


In [4]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-B  │  Working directory & path configuration
# Run this cell FIRST after every kernel restart.
# ═══════════════════════════════════════════════════════════════════════════════

import os, sys

REPO_ROOT = "/kaggle/working/introml-clara-implementation"
assert os.path.isdir(REPO_ROOT), (
    f"Repository not found at {REPO_ROOT}. "
    "Please run Cell 0-A first, then restart the kernel."
)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Working directory : {os.getcwd()}")
print(f"Python path entry : {sys.path[0]}")

Working directory : /kaggle/working/introml-clara-implementation
Python path entry : /kaggle/working/introml-clara-implementation


In [5]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-C  │  GPU / VRAM diagnostics + verify checkpoint
# ═══════════════════════════════════════════════════════════════════════════════

import os, torch

APPLE_CKPT = "/kaggle/input/datasets/tokiggle/clara-7b-e2e-4q"

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable GPU: Settings > Accelerator > T4 or P100.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU       : {gpu_name}")
print(f"VRAM      : {vram_gb:.1f} GB")
print(f"CUDA      : {torch.version.cuda}")
print(f"PyTorch   : {torch.__version__}")
print(f"Checkpoint: {APPLE_CKPT}")

assert os.path.isdir(APPLE_CKPT), (
    f"Checkpoint not found at {APPLE_CKPT}. "
    "Attach the tokiggle/clara-7b-e2e-4q dataset in Kaggle settings."
)

print(f"Files     : {os.listdir(APPLE_CKPT)}")
print("\n✓ Environment ready.")

GPU       : Tesla T4
VRAM      : 15.6 GB
CUDA      : 12.1
PyTorch   : 2.3.1+cu121
Checkpoint: /kaggle/input/datasets/tokiggle/clara-7b-e2e-4q
Files     : ['config.json', 'tokenizer.json', 'tokenizer_config.json', 'modeling_clara.py', 'chat_template.jinja', 'decoder_first_last_layers.pth', 'special_tokens_map.json', 'tokenizer.model', 'adapters.pth']

✓ Environment ready.


---
## Section 1 — Quick Test (5 SQuAD samples)

Run a fast evaluation on 5 samples to verify the Apple native pipeline produces sensible answers.  
This uses `scripts/evaluate_apple.py` via subprocess, same pattern as the main evaluation notebook.

In [6]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1  │  Quick Test: Apple Native Eval (5 SQuAD samples)
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

APPLE_CKPT = "/kaggle/input/datasets/tokiggle/clara-7b-e2e-4q"

print("╔" + "═" * 60 + "╗")
print("║  QUICK TEST — Apple Native Pipeline (5 SQuAD samples)       ║")
print("╠" + "═" * 60 + "╣")
print("║  Uses Apple's own modeling_clara.py (4-bit NF4)             ║")
print("║  generate_from_questions() E2E stage2 pipeline              ║")
print("╚" + "═" * 60 + "╝")

eval_env = os.environ.copy()
eval_env.update({
    "CLARA_CKPT_PATH"      : APPLE_CKPT,
    "CLARA_DATASET"        : "squad",
    "CLARA_EVAL_MODE"      : "oracle",
    "CLARA_EVAL_BS"        : "1",
    "CLARA_N_VAL"          : "5",
    "CLARA_MAX_NEW_TOKENS" : "32",
    "CLARA_MODEL_VERSION"  : "QuickTest_AppleNative",
})

subprocess.run(
    ["python", "-m", "scripts.evaluate_apple"],
    env=eval_env, check=True,
)

print("\n✅ Quick test complete!")
print("If predictions look sensible → proceed to Section 2 or 3.")

╔════════════════════════════════════════════════════════════╗
║  QUICK TEST — Apple Native Pipeline (5 SQuAD samples)       ║
╠════════════════════════════════════════════════════════════╣
║  Uses Apple's own modeling_clara.py (4-bit NF4)             ║
║  generate_from_questions() E2E stage2 pipeline              ║
╚════════════════════════════════════════════════════════════╝


/root/.cache/huggingface/modules/transformers_modules/apple-eval-workdir/modeling_clara.py:1039: SyntaxWarning: invalid escape sequence '\ '
  user_prompt = [{"role": "user", "content": prompt_user.replace(':\ ', ': ')}]
/root/.cache/huggingface/modules/transformers_modules/apple-eval-workdir/modeling_clara.py:1082: SyntaxWarning: invalid escape sequence '\ '
  user_prompt = [{"role": "user", "content": prompt_user.replace(':\ ', ': ')}]
/root/.cache/huggingface/modules/transformers_modules/apple-eval-workdir/modeling_clara.py:1104: SyntaxWarning: invalid escape sequence '\ '
  combined_content = prompt_system + '\n' + prompt_user.replace(':\ ', ': ')
/root/.cache/huggingface/modules/transformers_modules/apple-eval-workdir/modeling_clara.py:1127: SyntaxWarning: invalid escape sequence '\ '
  user_prompt = [{"role": "user", "content": prompt_user.replace(':\ ', ': ')}]
/root/.cache/huggingface/modules/transformers_modules/apple-eval-workdir/modeling_clara.py:1149: SyntaxWarning: invalid

CLaRa Evaluation — Apple Native Pipeline
  Checkpoint : /kaggle/input/datasets/tokiggle/clara-7b-e2e-4q
  Dataset    : squad
  Eval mode  : oracle
  Batch size : 1
  Val samples: 5
  Max tokens : 32

[1/4] Assembling workdir...
  ✓ Workdir: /kaggle/working/apple-eval-workdir
    - Source: /kaggle/input/datasets/tokiggle/clara-7b-e2e-4q
    - Patched: quantization=int4, decoder=mistralai/Mistral-7B-Instruct-v0.2

[2/4] Loading Apple CLaRa model (4-bit NF4)...
Initializing model from trained checkpoint: CLaRaConfig {
  "_attn_implementation_autoset": true,
  "ae_mode": "token",
  "attn_implementation": null,
  "auto_map": {
    "AutoConfig": "modeling_clara.CLaRaConfig",
    "AutoModel": "modeling_clara.CLaRa"
  },
  "compr_base_model_name": "mistralai/Mistral-7B-Instruct-v0.2",
  "compr_every_n_layer": null,
  "compr_linear_type": "concat",
  "compr_mlp_hidden_dim": 8096,
  "compr_model_name": null,
  "compr_n_layers": 5,
  "compr_rate": 16,
  "compr_rms_norm": false,
  "compr_use_mlp":

Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.16s/it]


Base decoder parameters: 7241732096
Model adapter keys: []
Memory token count: 16
Loading checkpoint adapter: decoder_adapter
Loading checkpoint adapter: encoder_adapter
Loading checkpoint adapter: query_reasoner_adapter
  ✓ Model loaded. VRAM: 5.0/15.6 GB

[3/4] Loading 'squad' validation set...


Evaluating:   0%|          | 0/5 [00:00<?, ?batch/s]We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


  ✓ Loaded 5 samples from squad (eval_mode=oracle)

[4/4] Running evaluation...


Evaluating: 100%|██████████| 5/5 [00:28<00:00,  5.66s/batch]



✅ Results saved to: results/eval_scores.csv

EVALUATION RESULTS (Apple Native Pipeline)
  Dataset    : squad  (eval_mode=oracle)
  Samples    : 5
  Exact Match: 80.00%
  F1 Score   : 80.00%

Sample predictions (first 5):
  Gold : Denver Broncos
  Pred : Denver Broncos
  EM=1  F1=1.00
  --------------------------------------------------
  Gold : Carolina Panthers
  Pred : Carolina Panthers
  EM=1  F1=1.00
  --------------------------------------------------
  Gold : Santa Clara, California
  Pred : Levi's Stadium
  EM=0  F1=0.00
  --------------------------------------------------
  Gold : Denver Broncos
  Pred : Denver Broncos
  EM=1  F1=1.00
  --------------------------------------------------
  Gold : gold
  Pred : gold
  EM=1  F1=1.00
  --------------------------------------------------

✅ Quick test complete!
If predictions look sensible → proceed to Section 2 or 3.


---
## Section 2 — Manual Inference (Hand-Crafted Examples)

Load the model directly in-notebook and run custom Q&A examples for qualitative inspection.

In [7]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2  │  Manual Inference: Load model + run custom Q&A examples
# ═══════════════════════════════════════════════════════════════════════════════

import torch, gc, os
from transformers import AutoModel
from scripts.evaluate_apple import assemble_workdir

MANUAL_INFERENCE = False

if MANUAL_INFERENCE:
    APPLE_CKPT = "/kaggle/input/datasets/tokiggle/clara-7b-e2e-4q"
    
    # ── Load model ────────────────────────────────────────────────────────────────
    work_dir = assemble_workdir(APPLE_CKPT)
    
    gc.collect(); torch.cuda.empty_cache()
    model = AutoModel.from_pretrained(
        work_dir, trust_remote_code=True, load_pretrained_checkpoint=True,
    )
    model.to("cuda")
    print(f"VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
    
    # ── Test cases ────────────────────────────────────────────────────────────────
    TEST_CASES = [
        {
            'docs': [
                'The Battle of Hastings was fought on 14 October 1066 between '
                'the Norman-French army of William, the Duke of Normandy, and '
                'an English army under the Anglo-Saxon King Harold Godwinson.',
            ],
            'q': 'When was the Battle of Hastings fought?',
            'expected': '14 October 1066',
        },
        {
            'docs': [
                'Weldenia is a monotypic genus of flowering plants in the family '
                'Commelinaceae, native to Mexico and Guatemala.',
            ],
            'q': 'Which genus grows originally in Mexico and Guatemala, '
                 'Phylica or Weldenia?',
            'expected': 'Weldenia',
        },
        {
            'docs': [
                'Albert Einstein was born on 14 March 1879 in Ulm, in the '
                'Kingdom of Württemberg in the German Empire. He developed the '
                'theory of relativity.',
            ],
            'q': 'Where was Albert Einstein born?',
            'expected': 'Ulm',
        },
        {
            'docs': [
                'Python is a high-level, general-purpose programming language. '
                'Its design philosophy emphasizes code readability. '
                'Python was conceived in the late 1980s by Guido van Rossum.',
            ],
            'q': 'Who created Python?',
            'expected': 'Guido van Rossum',
        },
    ]
    
    print("═" * 65)
    print("  MANUAL INFERENCE TEST  (Apple E2E pipeline)")
    print("═" * 65)
    
    with torch.no_grad():
        for i, tc in enumerate(TEST_CASES, 1):
            decoded, _ = model.generate_from_questions(
                questions=[tc['q']],
                documents=[tc['docs']],
                max_new_tokens=64,
            )
            pred = decoded[0]
            match = tc['expected'].lower() in pred.lower()
    
            print(f"\n[{i}] Question : {tc['q']}")
            print(f"    Expected : {tc['expected']}")
            print(f"    Model    : {pred[:200]}")
            print(f"    Match    : {'✓ YES' if match else '✗ NO'}")
    
    print("\n" + "═" * 65)
    
    # Free memory before full eval
    del model; gc.collect(); torch.cuda.empty_cache()
    print("Model unloaded. Ready for full evaluation.")

---
## Section 3 — Full Evaluation (TriviaQA + SQuAD)

Evaluates the Apple CLaRa-7B-E2E checkpoint on 500 validation samples from each dataset.  
Results are saved to `results/eval_scores.csv`.

In [8]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3  │  Full Apple Native Eval — TriviaQA + SQuAD (500 samples each)
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

APPLE_CKPT = "/kaggle/input/datasets/tokiggle/clara-7b-e2e-4q"

for dataset in ["triviaqa", "squad"]:
    print("\n" + "╔" + "═" * 60 + "╗")
    print(f"║  MODEL A — Apple Native Eval: {dataset.upper():<30}║")
    print("╠" + "═" * 60 + "╣")
    print("║  Pipeline  : Apple modeling_clara.py (4-bit NF4)            ║")
    print(f"║  Dataset   : {dataset:<47}║")
    print("║  Eval mode : oracle  |  Metrics: EM + F1                    ║")
    print("╚" + "═" * 60 + "╝")

    eval_env = os.environ.copy()
    eval_env.update({
        "CLARA_CKPT_PATH"      : APPLE_CKPT,
        "CLARA_DATASET"        : dataset,
        "CLARA_EVAL_MODE"      : "oracle",
        "CLARA_EVAL_BS"        : "1",
        "CLARA_N_VAL"          : "500",
        "CLARA_MAX_NEW_TOKENS" : "32",
        "CLARA_MODEL_VERSION"  : f"ModelA_AppleNative_{dataset}",
    })

    subprocess.run(
        ["python", "-m", "scripts.evaluate_apple"],
        env=eval_env, check=True,
    )

print("\n✅ Apple native evaluation complete (both datasets).")
print("Results appended to: results/eval_scores.csv")


╔════════════════════════════════════════════════════════════╗
║  MODEL A — Apple Native Eval: TRIVIAQA                      ║
╠════════════════════════════════════════════════════════════╣
║  Pipeline  : Apple modeling_clara.py (4-bit NF4)            ║
║  Dataset   : triviaqa                                       ║
║  Eval mode : oracle  |  Metrics: EM + F1                    ║
╚════════════════════════════════════════════════════════════╝


/root/.cache/huggingface/modules/transformers_modules/apple-eval-workdir/modeling_clara.py:1039: SyntaxWarning: invalid escape sequence '\ '
  user_prompt = [{"role": "user", "content": prompt_user.replace(':\ ', ': ')}]
/root/.cache/huggingface/modules/transformers_modules/apple-eval-workdir/modeling_clara.py:1082: SyntaxWarning: invalid escape sequence '\ '
  user_prompt = [{"role": "user", "content": prompt_user.replace(':\ ', ': ')}]
/root/.cache/huggingface/modules/transformers_modules/apple-eval-workdir/modeling_clara.py:1104: SyntaxWarning: invalid escape sequence '\ '
  combined_content = prompt_system + '\n' + prompt_user.replace(':\ ', ': ')
/root/.cache/huggingface/modules/transformers_modules/apple-eval-workdir/modeling_clara.py:1127: SyntaxWarning: invalid escape sequence '\ '
  user_prompt = [{"role": "user", "content": prompt_user.replace(':\ ', ': ')}]
/root/.cache/huggingface/modules/transformers_modules/apple-eval-workdir/modeling_clara.py:1149: SyntaxWarning: invalid

CLaRa Evaluation — Apple Native Pipeline
  Checkpoint : /kaggle/input/datasets/tokiggle/clara-7b-e2e-4q
  Dataset    : triviaqa
  Eval mode  : oracle
  Batch size : 1
  Val samples: 500
  Max tokens : 32

[1/4] Assembling workdir...
  ✓ Workdir: /kaggle/working/apple-eval-workdir
    - Source: /kaggle/input/datasets/tokiggle/clara-7b-e2e-4q
    - Patched: quantization=int4, decoder=mistralai/Mistral-7B-Instruct-v0.2

[2/4] Loading Apple CLaRa model (4-bit NF4)...
  ✓ Cleared stale HF module cache
Initializing model from trained checkpoint: CLaRaConfig {
  "_attn_implementation_autoset": true,
  "ae_mode": "token",
  "attn_implementation": null,
  "auto_map": {
    "AutoConfig": "modeling_clara.CLaRaConfig",
    "AutoModel": "modeling_clara.CLaRa"
  },
  "compr_base_model_name": "mistralai/Mistral-7B-Instruct-v0.2",
  "compr_every_n_layer": null,
  "compr_linear_type": "concat",
  "compr_mlp_hidden_dim": 8096,
  "compr_model_name": null,
  "compr_n_layers": 5,
  "compr_rate": 16,
  "com

Loading checkpoint shards: 100%|██████████| 3/3 [00:35<00:00, 11.98s/it]


Base decoder parameters: 7241732096
Model adapter keys: []
Memory token count: 16
Loading checkpoint adapter: decoder_adapter
Loading checkpoint adapter: encoder_adapter
Loading checkpoint adapter: query_reasoner_adapter
  ✓ Model loaded. VRAM: 5.0/15.6 GB

[3/4] Loading 'triviaqa' validation set...


...

CLaRa Evaluation — Apple Native Pipeline
  Checkpoint : /kaggle/input/datasets/tokiggle/clara-7b-e2e-4q
  Dataset    : squad
  Eval mode  : oracle
  Batch size : 1
  Val samples: 500
  Max tokens : 32

[1/4] Assembling workdir...
  ✓ Workdir: /kaggle/working/apple-eval-workdir
    - Source: /kaggle/input/datasets/tokiggle/clara-7b-e2e-4q
    - Patched: quantization=int4, decoder=mistralai/Mistral-7B-Instruct-v0.2

[2/4] Loading Apple CLaRa model (4-bit NF4)...
  ✓ Cleared stale HF module cache
Initializing model from trained checkpoint: CLaRaConfig {
  "_attn_implementation_autoset": true,
  "ae_mode": "token",
  "attn_implementation": null,
  "auto_map": {
    "AutoConfig": "modeling_clara.CLaRaConfig",
    "AutoModel": "modeling_clara.CLaRa"
  },
  "compr_base_model_name": "mistralai/Mistral-7B-Instruct-v0.2",
  "compr_every_n_layer": null,
  "compr_linear_type": "concat",
  "compr_mlp_hidden_dim": 8096,
  "compr_model_name": null,
  "compr_n_layers": 5,
  "compr_rate": 16,
  "compr_

Loading checkpoint shards: 100%|██████████| 3/3 [00:06<00:00,  2.13s/it]


Base decoder parameters: 7241732096
Model adapter keys: []
Memory token count: 16
Loading checkpoint adapter: decoder_adapter
Loading checkpoint adapter: encoder_adapter
Loading checkpoint adapter: query_reasoner_adapter
  ✓ Model loaded. VRAM: 5.0/15.6 GB

[3/4] Loading 'squad' validation set...
  ✓ Loaded 500 samples from squad (eval_mode=oracle)

[4/4] Running evaluation...


Evaluating: 100%|██████████| 500/500 [51:04<00:00,  6.13s/batch]



✅ Results saved to: results/eval_scores.csv

EVALUATION RESULTS (Apple Native Pipeline)
  Dataset    : squad  (eval_mode=oracle)
  Samples    : 500
  Exact Match: 75.40%
  F1 Score   : 81.64%

Sample predictions (first 5):
  Gold : Denver Broncos
  Pred : Denver Broncos
  EM=1  F1=1.00
  --------------------------------------------------
  Gold : Carolina Panthers
  Pred : Carolina Panthers
  EM=1  F1=1.00
  --------------------------------------------------
  Gold : Santa Clara, California
  Pred : Levi's Stadium
  EM=0  F1=0.00
  --------------------------------------------------
  Gold : Denver Broncos
  Pred : Denver Broncos
  EM=1  F1=1.00
  --------------------------------------------------
  Gold : gold
  Pred : gold
  EM=1  F1=1.00
  --------------------------------------------------

✅ Apple native evaluation complete (both datasets).
Results appended to: results/eval_scores.csv


---
## Section 4 — Results Summary

In [9]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4  │  Results Summary
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pandas as pd

CSV_PATH = "results/eval_scores.csv"

if not os.path.exists(CSV_PATH):
    print(f"No results found at {CSV_PATH}. Run evaluation cells first.")
else:
    df = pd.read_csv(CSV_PATH)

    print("═" * 80)
    print("  CLaRa EXPERIMENT RESULTS SUMMARY")
    print("═" * 80)
    print()

    display_df = df[[
        'Model_Version', 'Dataset', 'Eval_Mode',
        'Exact_Match(%)', 'F1_Score(%)', 'Timestamp'
    ]].copy()
    display_df = display_df.sort_values(['Dataset', 'Model_Version'])

    pd.set_option('display.max_colwidth', 45)
    pd.set_option('display.width', 120)
    print(display_df.to_string(index=False))

    print()
    print("─" * 80)
    print("PAPER REFERENCE (Table 2 — Oracle, CLaRa-Mistral-7B 16×):")
    print("  NQ        : EM=63.29%  F1=71.54%")
    print("  HotpotQA  : EM=57.54%  F1=71.17%")
    print("  (Instruction-tuned init, Normal setting)")
    print("─" * 80)

════════════════════════════════════════════════════════════════════════════════
  CLaRa EXPERIMENT RESULTS SUMMARY
════════════════════════════════════════════════════════════════════════════════

              Model_Version  Dataset Eval_Mode  Exact_Match(%)  F1_Score(%)           Timestamp
   ModelA_AppleNative_squad    squad    oracle            75.4        81.64 2026-05-03 19:06:45
      QuickTest_AppleNative    squad    oracle            80.0        80.00 2026-05-03 17:23:53
ModelA_AppleNative_triviaqa triviaqa    oracle            48.0        53.59 2026-05-03 18:15:23

────────────────────────────────────────────────────────────────────────────────
PAPER REFERENCE (Table 2 — Oracle, CLaRa-Mistral-7B 16×):
  NQ        : EM=63.29%  F1=71.54%
  HotpotQA  : EM=57.54%  F1=71.17%
  (Instruction-tuned init, Normal setting)
────────────────────────────────────────────────────────────────────────────────


---
## Experimental Notes

### Why Apple Native Pipeline?

The repo's `models/clara_model.py` had critical architectural mismatches:

1. **Compression**: Used embedding concatenation instead of memory token injection
2. **Adapter names**: `compressor/query/generator` vs Apple's `encoder_adapter/query_reasoner_adapter/decoder_adapter`
3. **LoRA targets**: Attention-only vs `all-linear` (drops ~50% of trained weights)
4. **Prompt format**: Missing system prompt and memory token placeholders
5. **Special tokens**: `decoder_first_last_layers.pth` embeddings not loaded

Using Apple's original code with `trust_remote_code=True` bypasses all of these.

### T4 GPU Adaptations

| Setting | Paper | This notebook |
|---------|-------|---------------|
| Quantization | BF16 (8×H100) | NF4 4-bit (T4) |
| `generation_top_k` | 5 | 5 (same) |
| `doc_max_length` | 256 | 256 (same) |

### Citation

```bibtex
@article{he2026clara,
  title   = {CLaRa: Bridging Retrieval and Generation with Continuous Latent Reasoning},
  author  = {He, Jie and Bai, Richard He and Williamson, Sinead and Pan, Jeff Z. 
             and Jaitly, Navdeep and Zhang, Yizhe},
  journal = {arXiv preprint arXiv:2511.18659},
  year    = {2026}
}
```